# Trackpoint Analysis

**Purpose:** load tracker + GPS data, clean it, and visualize truck paths — optionally overlaid on mine zone polygons. This is the general-purpose exploration notebook; route extraction and clustering live in `route_clustering_analysis.ipynb`.

In [ ]:
import sys
sys.path.append("../..")

from gps_lib import io_utils, classify, preprocess, plotting

In [ ]:
tracker_list_df = io_utils.load_tracker_list()
tracker_list_df = classify.classify_technic_material_type(tracker_list_df)
tracker_list_df.groupby(["technic_type", "technic_m_type"])["label"].count()

In [ ]:
track_points_df = io_utils.load_gps_data("gps_data.csv")
track_points_df.shape

In [ ]:
merged_df = preprocess.attach_technic_info(track_points_df, tracker_list_df)

START_TIME = "2025-11-01 00:00:00"
END_TIME = "2025-11-04 10:00:00"
filtered_df = preprocess.filter_by_time(merged_df, START_TIME, END_TIME)
filtered_df[["tracker_id", "label", "get_time", "lat", "lng", "technic_type"]].head()

In [ ]:
df_cleaned = preprocess.clean_gps_points(filtered_df, round_n=4)
print(f"Removed {filtered_df.shape[0] - df_cleaned.shape[0]} near-duplicate points")
df_cleaned.shape

In [ ]:
display(plotting.list_technics(df_cleaned, technic_type="dump", max_rows=10))
plotting.plot_tracker_paths(df_cleaned, technic_idx=0, technic_type="dump")

In [ ]:
# Overlay with mine zones (requires data/zone_list.csv + data/zone_detail_all_df.csv,
# produced by notebooks/api/fetch_zones.ipynb)
try:
    zone_list_df = io_utils.load_zone_list()
    zone_detail_all_df = io_utils.load_zone_detail()
    zone_filtered_df = classify.classify_zones(zone_list_df, drop_other=True)

    plotting.plot_zones_with_tracker_paths(
        zone_df=zone_detail_all_df,
        zone_meta_df=zone_filtered_df,
        tracker_df=df_cleaned,
        material_types=["bn"],
        load_types=["load", "unload"],
        n=10,
        technic_type="dump",
        technic_m_type="bn",
        figsize=(14, 12),
    )
except FileNotFoundError as e:
    print(f"Zone data files not found: {e}")
    print("Run notebooks/api/fetch_zones.ipynb first.")

In [ ]:
# Date coverage summary
unique_dates = sorted(df_cleaned["date"].unique())
print(f"Total unique days: {len(unique_dates)}")
print(f"Range: {df_cleaned['date'].min()} -> {df_cleaned['date'].max()}")
for date in unique_dates:
    print(f"  {date}: {(df_cleaned['date'] == date).sum():,} points")